In [0]:
import time, mlflow, mlflow.spark
from pyspark.ml.classification import (LogisticRegression, DecisionTreeClassifier,
    RandomForestClassifier, GBTClassifier, LinearSVC)
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

train_p = spark.read.parquet("/Volumes/workspace/default/fema_claims_project/train_prepared/")
val_p = spark.read.parquet("/Volumes/workspace/default/fema_claims_project/val_prepared/")

label = "high_payout"
bin_eval = BinaryClassificationEvaluator(labelCol=label, metricName="areaUnderROC")
mc_eval = MulticlassClassificationEvaluator(labelCol=label, predictionCol="prediction", metricName="accuracy")

models = {
    "LogisticRegression": LogisticRegression(featuresCol="features", labelCol=label, maxIter=100),
    "DecisionTree": DecisionTreeClassifier(featuresCol="features", labelCol=label, maxDepth=10),
    "RandomForest": RandomForestClassifier(featuresCol="features", labelCol=label, numTrees=100),
    "GBT": GBTClassifier(featuresCol="features", labelCol=label, maxIter=100),
    "LinearSVC": LinearSVC(featuresCol="features", labelCol=label, maxIter=100),
}

# MLflow requires a Unity Catalog volume path for dfs_tmpdir in shared/serverless clusters.
mlflow_tmp_dir = "/Volumes/workspace/default/fema_claims_project/mlflow_tmp"

cls_results = []
for name, model in models.items():
    with mlflow.start_run(run_name=f"CLS_{name}"):
        t0 = time.time()
        fitted = model.fit(train_p)
        duration = time.time() - t0
        preds = fitted.transform(val_p)
        acc = mc_eval.evaluate(preds)
        # AUC only works for models with probability column
        try: auc = bin_eval.evaluate(preds)
        except: auc = 0.0
        mlflow.log_metrics({"accuracy": acc, "auc_roc": auc, "time_s": duration})
        # Pass dfs_tmpdir argument to log_model to specify the temporary directory for MLflow
        mlflow.spark.log_model(fitted, name, dfs_tmpdir=mlflow_tmp_dir)
        cls_results.append((name, acc, auc, duration))
        print(f"✅ {name:20s} → Acc: {acc:.4f} | AUC: {auc:.4f} | {duration:.1f}s")

cls_df = spark.createDataFrame(cls_results, ["model","accuracy","auc","time_s"])
display(cls_df)

2026/03/02 21:48:30 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.0.0+databricks.connect.17.3.2) contains a local version label (+databricks.connect.17.3.2). MLflow logged a pip requirement for this package as 'pyspark==4.0.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/03/02 21:48:35 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /local_disk0/user_tmp_data/spark-3eaf8640-cd3f-453f-9ca4-4c/tmp29r_9_z2/model, flavor: spark). Fall back to return ['pyspark==4.0.0']. Set logging level to DEBUG to see the full traceback. 
2026/03/02 21:48:35 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


✅ LogisticRegression   → Acc: 0.9772 | AUC: 0.9972 | 10.1s


2026/03/02 21:48:47 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.0.0+databricks.connect.17.3.2) contains a local version label (+databricks.connect.17.3.2). MLflow logged a pip requirement for this package as 'pyspark==4.0.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/03/02 21:48:50 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /local_disk0/user_tmp_data/spark-3eaf8640-cd3f-453f-9ca4-4c/tmprnqktfn3/model, flavor: spark). Fall back to return ['pyspark==4.0.0']. Set logging level to DEBUG to see the full traceback. 
2026/03/02 21:48:50 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


✅ DecisionTree         → Acc: 0.9978 | AUC: 0.9998 | 3.2s


2026/03/02 21:49:36 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.0.0+databricks.connect.17.3.2) contains a local version label (+databricks.connect.17.3.2). MLflow logged a pip requirement for this package as 'pyspark==4.0.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/03/02 21:49:38 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /local_disk0/user_tmp_data/spark-3eaf8640-cd3f-453f-9ca4-4c/tmp3p1ow0hy/model, flavor: spark). Fall back to return ['pyspark==4.0.0']. Set logging level to DEBUG to see the full traceback. 
2026/03/02 21:49:38 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


...

2026/03/02 21:52:25 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.0.0+databricks.connect.17.3.2) contains a local version label (+databricks.connect.17.3.2). MLflow logged a pip requirement for this package as 'pyspark==4.0.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/03/02 21:52:28 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /local_disk0/user_tmp_data/spark-3eaf8640-cd3f-453f-9ca4-4c/tmpcfirpjhq/model, flavor: spark). Fall back to return ['pyspark==4.0.0']. Set logging level to DEBUG to see the full traceback. 
2026/03/02 21:52:28 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


✅ GBT                  → Acc: 0.9980 | AUC: 1.0000 | 146.6s


2026/03/02 21:53:03 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.0.0+databricks.connect.17.3.2) contains a local version label (+databricks.connect.17.3.2). MLflow logged a pip requirement for this package as 'pyspark==4.0.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/03/02 21:53:05 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /local_disk0/user_tmp_data/spark-3eaf8640-cd3f-453f-9ca4-4c/tmpi60r4w53/model, flavor: spark). Fall back to return ['pyspark==4.0.0']. Set logging level to DEBUG to see the full traceback. 
2026/03/02 21:53:05 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


✅ LinearSVC            → Acc: 0.9780 | AUC: 0.9972 | 26.9s


model,accuracy,auc,time_s
LogisticRegression,0.977234026426821,0.9971576795465582,10.061434507369995
DecisionTree,0.997847452060332,0.999774980049798,3.2476606369018555
RandomForest,0.9853060812005761,0.9988974691402376,24.900749683380127
GBT,0.9980116722669968,0.9999602137356356,146.61340188980103
LinearSVC,0.9779515424067103,0.9971699216956357,26.85383915901184


In [0]:
from pyspark.ml.clustering import KMeans, BisectingKMeans, GaussianMixture
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml.feature import PCA, StandardScaler
from pyspark.ml import Pipeline
from pyspark import StorageLevel
from pyspark.sql.types import StructType, StructField, StringType, FloatType
import time

# ═══════════════════════════════════════════════════════════════════════════
# CRITICAL FIRST STEP: Check your feature dimensionality
# GMM memory cost = O(k * d²) — if d > 15, GMM will OOM on Community Edition
# ═══════════════════════════════════════════════════════════════════════════
sample_row = train_p.select("features").limit(1).collect()
feature_dim = len(sample_row[0]["features"])
print(f"⚠️  Feature dimensionality: {feature_dim}")
print(f"   GMM covariance memory estimate: k * d² = 2 * {feature_dim}² = {2 * feature_dim**2} values per iteration")

# ═══════════════════════════════════════════════════════════════════════════
# AGGRESSIVE MEMORY STRATEGY:
# 1. Tiny sample (1%) cached in MEMORY — not DISK
# 2. Reduce to 4 PCA components regardless of original dimensionality
# 3. Each model: fresh unpersist → recompute → unpersist cycle
# 4. Spark cache cleared completely between models
# 5. GMM isolated to PCA features with maxIter=3 (minimum viable)
# ═══════════════════════════════════════════════════════════════════════════

SAMPLE_FRACTION = 1.0   # 1% — ~27,000 rows from 2.7M
PCA_DIMS        = 4      # Keeps GMM covariance matrix tiny: 2 * 4² = 32 values
K               = 2
SEED            = 42

# Step 1: Sample and cache in MEMORY to avoid recomputation without RAM pressure
# NOTE: Only MEMORY caching is supported on serverless compute; DISK persistence is not allowed.
clu_sample = (
    train_p
    .sample(withReplacement=False, fraction=SAMPLE_FRACTION, seed=SEED)
    .repartition(4)       # Small dataset — fewer partitions = less overhead
    #.cache()              # MEMORY_ONLY caching for serverless compatibility
)
n_sample = clu_sample.count()
print(f"✅ Sample size: {n_sample:,} rows")

# Step 2: Fit PCA on sample and cache PCA-transformed data in memory
pca_pipe = PCA(k=PCA_DIMS, inputCol="features", outputCol="pcaFeatures").fit(clu_sample)
clu_pca = (
    pca_pipe.transform(clu_sample)
    .select("pcaFeatures")
   # .cache()              # MEMORY_ONLY caching for serverless compatibility
)
clu_pca.count()
print(f"✅ PCA variance explained: {sum(pca_pipe.explainedVariance):.3f}")

# ═══════════════════════════════════════════════════════════════════════════
# Model runner — fully isolated per model with cache wipe between runs
# ═══════════════════════════════════════════════════════════════════════════
def run_clustering_model(name, model, data, features_col):
    """
    Runs a single clustering model in full isolation.
    Cache clearing is skipped due to serverless limitations.
    """
    print(f"\n{'═'*55}")
    print(f"  ▶ Running: {name}")
    print(f"{'═'*55}")

    with mlflow.start_run(run_name=f"CLU_{name}"):
        try:
            # NOTE: Cache clearing skipped on serverless
            t0 = time.time()
            fitted = model.fit(data)
            duration = time.time() - t0
            print(f"  ✅ Fit complete in {duration:.1f}s")

            preds = fitted.transform(data)

            evaluator = ClusteringEvaluator(
                featuresCol=features_col,
                predictionCol="cluster",
                metricName="silhouette"
            )
            sil = evaluator.evaluate(preds)
            print(f"  ✅ Silhouette: {sil:.4f}")

            mlflow.log_params({
                "algorithm":       name,
                "k":               K,
                "sample_fraction": SAMPLE_FRACTION,
                "pca_dims":        PCA_DIMS if "pca" in features_col.lower() else "none"
            })
            mlflow.log_metrics({
                "silhouette":       sil,
                "training_time_s":  round(duration, 2)
            })
            mlflow.spark.log_model(fitted, name)

            return (name, round(sil, 4), round(duration, 2), "success")

        except Exception as e:
            print(f"  ❌ FAILED: {e}")
            mlflow.log_param("status", "failed")
            return (name, None, None, "failed")

        finally:
            # NOTE: Cache clearing skipped on serverless
            pass

# ═══════════════════════════════════════════════════════════════════════════
# Run each model — note GMM uses PCA features to keep covariance matrix small
# ═══════════════════════════════════════════════════════════════════════════
clu_results = []

# KMeans on original features
result = run_clustering_model(
    "KMeans",
    KMeans(featuresCol="features", predictionCol="cluster", k=K, seed=SEED, maxIter=10),
    clu_sample,
    "features"
)
clu_results.append(result)

# BisectingKMeans on original features
result = run_clustering_model(
    "BisectingKMeans",
    BisectingKMeans(featuresCol="features", predictionCol="cluster", k=K, maxIter=10, seed=SEED),
    clu_sample,
    "features"
)
clu_results.append(result)

# GMM — MUST use PCA features; raw features will always OOM on Community Edition
result = run_clustering_model(
    "GaussianMixture",
    GaussianMixture(featuresCol="pcaFeatures", predictionCol="cluster", k=K, maxIter=3, seed=SEED),
    clu_pca,
    "pcaFeatures"
)
clu_results.append(result)

# PCA+KMeans
result = run_clustering_model(
    "PCA_KMeans",
    KMeans(featuresCol="pcaFeatures", predictionCol="cluster", k=K, seed=SEED, maxIter=10),
    clu_pca,
    "pcaFeatures"
)
clu_results.append(result)

# ═══════════════════════════════════════════════════════════════════════════
# Final cleanup and results
# ═══════════════════════════════════════════════════════════════════════════
#clu_sample.unpersist()
#clu_pca.unpersist()
# NOTE: spark.catalog.clearCache() is not supported on serverless compute; skipping cache clearing.

clu_schema = StructType([
    StructField("model", StringType(), True),
    StructField("silhouette", FloatType(), True),
    StructField("time_s", FloatType(), True),
    StructField("status", StringType(), True)
])

clu_df = spark.createDataFrame(
    [(r[0], r[1], r[2], r[3]) for r in clu_results],
    schema=clu_schema
)
display(clu_df)

⚠️  Feature dimensionality: 13
   GMM covariance memory estimate: k * d² = 2 * 13² = 338 values per iteration
✅ Sample size: 1,849,377 rows
✅ PCA variance explained: 0.638

═══════════════════════════════════════════════════════
  ▶ Running: KMeans
═══════════════════════════════════════════════════════
  ✅ Fit complete in 11.4s
  ✅ Silhouette: 0.9999
  ❌ FAILED: UC volume path must be provided to save, log or load SparkML models in Databricks shared or serverless clusters. Specify environment variable 'MLFLOW_DFS_TMP' or 'dfs_tmpdir' argument that uses a UC volume path starting with '/Volumes/...' when saving, logging or loading a model.

═══════════════════════════════════════════════════════
  ▶ Running: BisectingKMeans
═══════════════════════════════════════════════════════


{"ts": "2026-03-02 21:53:39.655", "level": "ERROR", "logger": "pyspark.sql.connect.logging", "msg": "GRPC Error received", "context": {}, "exception": {"class": "_MultiThreadedRendezvous", "msg": "<_MultiThreadedRendezvous of RPC that terminated with:\n\tstatus = StatusCode.INTERNAL\n\tdetails = \"[DBFS_DISABLED] Public DBFS root is disabled. Access is denied on path: /local_disk0/tmp/spark_connect_model_cache/2530d00c-290c-4337-a33a-e2f9e65b7913/9ee55690-f705-43fc-8d6b-99c50b2e66f0/data/metadata/_delta_log SQLSTATE: 56038\"\n\tdebug_error_string = \"UNKNOWN:Error received from peer ipv4:127.0.0.1:7073 {created_time:\"2026-03-02T21:53:39.65422751+00:00\", grpc_status:13, grpc_message:\"[DBFS_DISABLED] Public DBFS root is disabled. Access is denied on path: /local_disk0/tmp/spark_connect_model_cache/2530d00c-290c-4337-a33a-e2f9e65b7913/9ee55690-f705-43fc-8d6b-99c50b2e66f0/data/metadata/_delta_log SQLSTATE: 56038\"}\"\n>", "stacktrace": [{"class": null, "method": "_execute_and_fetch_as_i

  ❌ FAILED: [DBFS_DISABLED] Public DBFS root is disabled. Access is denied on path: /local_disk0/tmp/spark_connect_model_cache/2530d00c-290c-4337-a33a-e2f9e65b7913/9ee55690-f705-43fc-8d6b-99c50b2e66f0/data/metadata/_delta_log SQLSTATE: 56038

JVM stacktrace:
com.databricks.backend.daemon.data.client.DbfsUnsupportedOperationSparkException
	at com.databricks.backend.daemon.data.client.DbfsExceptionMapperImpl.withExceptionWrapping(DbfsSparkException.scala:42)
	at com.databricks.backend.daemon.data.client.DBFSV2.getFileStatus(DatabricksFileSystemV2.scala:750)
	at com.databricks.backend.daemon.data.client.DatabricksFileSystem.getFileStatus(DatabricksFileSystem.scala:212)
	at com.databricks.sql.io.LokiFileSystem.$anonfun$getFileStatusNoCache$4(LokiFileSystem.scala:347)
	at com.databricks.sql.io.LokiFileSystem.tryWithNativeIO(LokiFileSystem.scala:319)
	at com.databricks.sql.io.LokiFileSystem.$anonfun$getFileStatusNoCache$1(LokiFileSystem.scala:347)
	at com.databricks.sql.io.LokiFileSystem$.wi

model,silhouette,time_s,status
KMeans,null,null,failed
BisectingKMeans,null,null,failed
GaussianMixture,null,null,failed
PCA_KMeans,null,null,failed
